In [19]:
# Import libraries
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score

In [20]:
# Read train, validation, and test data
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(processed_dir / "train.csv")
valid_df = pd.read_csv(processed_dir / "valid.csv")
test_df = pd.read_csv(processed_dir / "test.csv")
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test:", test_df.shape)
train_df.head()

Train: (43819, 28)
Valid: (8784, 28)
Test: (8757, 28)


,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,hour,month,...,cloud_cover_lag_2h,cloud_cover_lag_3h,temperature_2m_lag_1h,temperature_2m_lag_2h,temperature_2m_lag_3h,precipitation_rolling_3h,precipitation_rolling_6h,rain_next_1h,rain_next_2h,rain_next_3h
0,2019-01-01 05:00:00,0.0,64,1026.0,8.8,15.0,17,100,5,1,...,34.0,39.0,9.0,9.1,9.7,0.0,0.0,0,0,0
1,2019-01-01 06:00:00,0.0,64,1026.4,8.9,14.6,16,100,6,1,...,59.0,34.0,8.8,9.0,9.1,0.0,0.0,0,0,0
2,2019-01-01 07:00:00,0.0,71,1026.5,9.2,15.8,24,100,7,1,...,100.0,59.0,8.9,8.8,9.0,0.0,0.0,0,0,0
3,2019-01-01 08:00:00,0.0,67,1027.4,9.8,15.6,23,100,8,1,...,100.0,100.0,9.2,8.9,8.8,0.0,0.0,0,0,0
4,2019-01-01 09:00:00,0.0,65,1028.3,10.3,14.8,18,100,9,1,...,100.0,100.0,9.8,9.2,8.9,0.0,0.0,0,0,0


In [21]:
# Configure targets and threshold
target_cols = ["rain_next_1h", "rain_next_2h", "rain_next_3h"]
threshold = 0.5

target_cols

['rain_next_1h', 'rain_next_2h', 'rain_next_3h']

In [22]:
# Keep time variables raw, encode wind direction as sin/cos
def add_wind_direction_features(df):
    df = df.copy()
    df["wind_dir_sin"] = np.sin(2 * np.pi * df["wind_direction_10m"] / 360)
    df["wind_dir_cos"] = np.cos(2 * np.pi * df["wind_direction_10m"] / 360)
    return df

train_df = add_wind_direction_features(train_df)
valid_df = add_wind_direction_features(valid_df)
test_df = add_wind_direction_features(test_df)
train_valid_df = add_wind_direction_features(train_valid_df)

drop_cols = ["datetime", "wind_direction_10m"] + target_cols
feature_cols = [col for col in train_df.columns if col not in drop_cols]

print("num_features:", len(feature_cols))
feature_cols

num_features: 25


['precipitation',
 'relative_humidity_2m',
 'surface_pressure',
 'temperature_2m',
 'wind_speed_10m',
 'cloud_cover',
 'hour',
 'month',
 'dayofweek',
 'precipitation_lag_1h',
 'precipitation_lag_2h',
 'precipitation_lag_3h',
 'relative_humidity_2m_lag_1h',
 'relative_humidity_2m_lag_2h',
 'relative_humidity_2m_lag_3h',
 'cloud_cover_lag_1h',
 'cloud_cover_lag_2h',
 'cloud_cover_lag_3h',
 'temperature_2m_lag_1h',
 'temperature_2m_lag_2h',
 'temperature_2m_lag_3h',
 'precipitation_rolling_3h',
 'precipitation_rolling_6h',
 'wind_dir_sin',
 'wind_dir_cos']

In [23]:
# Split input features
X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]
X_train_valid = train_valid_df[feature_cols]

X_train.shape, X_valid.shape, X_test.shape

((43819, 25), (8784, 25), (8757, 25))

In [24]:
# Define model factory and evaluation function
def make_model(params):
    return RandomForestClassifier(
        **params,
        criterion="gini",
        bootstrap=True,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )


def evaluate_one_target(y_true, y_prob, y_pred, target, split_name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "model": "random_forest",
        "split": split_name,
        "target": target,
        "threshold": threshold,
        "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

In [25]:
# Tune hyperparameters on validation data
param_grid = [
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 5},
    {"n_estimators": 300, "max_depth": 8, "min_samples_leaf": 3},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 5},
    {"n_estimators": 400, "max_depth": 12, "min_samples_leaf": 3},
    {"n_estimators": 400, "max_depth": 16, "min_samples_leaf": 5},
    {"n_estimators": 500, "max_depth": 20, "min_samples_leaf": 5},
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 5},
]

tuning_rows = []
best_params_by_target = {}

for target in target_cols:
    best_score = float("inf")
    best_params = None

    for params in param_grid:
        print("Target:", target, "Params:", params)
        model = make_model(params)
        model.fit(X_train, train_df[target])

        valid_prob = model.predict_proba(X_valid)[:, 1]
        valid_pred = (valid_prob >= threshold).astype(int)
        metric = evaluate_one_target(valid_df[target], valid_prob, valid_pred, target, "valid")
        metric.update(params)
        tuning_rows.append(metric)

        if metric["log_loss"] < best_score:
            best_score = metric["log_loss"]
            best_params = params

    best_params_by_target[target] = best_params

tuning_df = pd.DataFrame(tuning_rows)
best_params_df = pd.DataFrame([
    {"target": target, **params}
    for target, params in best_params_by_target.items()
])

tuning_df.to_csv(results_dir / "random_forest_tuning_results.csv", index=False)

best_params_df

Target: rain_next_1h Params: {'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 5}
Target: rain_next_1h Params: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 3}
Target: rain_next_1h Params: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 5}
Target: rain_next_1h Params: {'n_estimators': 400, 'max_depth': 12, 'min_samples_leaf': 3}
Target: rain_next_1h Params: {'n_estimators': 400, 'max_depth': 16, 'min_samples_leaf': 5}
Target: rain_next_1h Params: {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 5}
Target: rain_next_1h Params: {'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 5}
Target: rain_next_2h Params: {'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 5}
Target: rain_next_2h Params: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 3}
Target: rain_next_2h Params: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 5}
Target: rain_next_2h Params: {'n_estimators': 400, 'max_depth': 12, 'min_samples_leaf': 3}
T

,target,n_estimators,max_depth,min_samples_leaf
0,rain_next_1h,400,NaN,5
1,rain_next_2h,500,20.0,5
2,rain_next_3h,400,NaN,5


In [26]:
# Train final models on train + valid and evaluate on test
final_models = {}
test_metric_rows = []
test_predictions = test_df[["datetime"]].copy()

for target in target_cols:
    params = best_params_by_target[target]
    print("Final target:", target, "Params:", params)

    model = make_model(params)
    model.fit(X_train_valid, train_valid_df[target])
    final_models[target] = model

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)
    metric = evaluate_one_target(test_df[target], test_prob, test_pred, target, "test")
    metric.update(params)
    test_metric_rows.append(metric)

    test_predictions[f"{target}_true"] = test_df[target].values
    test_predictions[f"{target}_prob"] = test_prob
    test_predictions[f"{target}_pred"] = test_pred

metrics_df = pd.DataFrame(test_metric_rows)
metrics_df = metrics_df[[
    "model", "split", "target", "threshold", "log_loss", "brier_score", "roc_auc", "pr_auc",
    "accuracy", "precision", "recall", "f1", "tn", "fp", "fn", "tp",
]]

metrics_df.to_csv(results_dir / "random_forest_test_metrics.csv", index=False)
test_predictions.to_csv(results_dir / "random_forest_test_predictions.csv", index=False)

metrics_df

Final target: rain_next_1h Params: {'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 5}
Final target: rain_next_2h Params: {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 5}
Final target: rain_next_3h Params: {'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 5}


,model,split,target,threshold,log_loss,brier_score,roc_auc,pr_auc,accuracy,precision,recall,f1,tn,fp,fn,tp
0,random_forest,test,rain_next_1h,0.5,0.301567,0.092235,0.930337,0.848221,0.877926,0.765323,0.781397,0.773277,5865,559,510,1823
1,random_forest,test,rain_next_2h,0.5,0.374084,0.120352,0.892438,0.756830,0.828252,0.659362,0.735105,0.695176,5538,886,618,1715
2,random_forest,test,rain_next_3h,0.5,0.396547,0.129050,0.876305,0.722139,0.807354,0.620522,0.712816,0.663475,5407,1017,670,1663


In [27]:
# Inspect the most important Random Forest features
feature_importance_rows = []

for target, model in final_models.items():
    for feature, importance in zip(feature_cols, model.feature_importances_):
        feature_importance_rows.append({
            "target": target,
            "feature": feature,
            "importance": importance,
        })

feature_importance_df = pd.DataFrame(feature_importance_rows)
feature_importance_df.sort_values(["target", "importance"], ascending=[True, False]).groupby("target").head(10)

,target,feature,importance
0,rain_next_1h,precipitation,0.251231
21,rain_next_1h,precipitation_rolling_3h,0.145334
22,rain_next_1h,precipitation_rolling_6h,0.107811
9,rain_next_1h,precipitation_lag_1h,0.074090
10,rain_next_1h,precipitation_lag_2h,0.047229
14,rain_next_1h,relative_humidity_2m_lag_3h,0.036690
11,rain_next_1h,precipitation_lag_3h,0.031317
1,rain_next_1h,relative_humidity_2m,0.023826
5,rain_next_1h,cloud_cover,0.023492
13,rain_next_1h,relative_humidity_2m_lag_2h,0.021302
